# Paired AdamW baseline versus AdamW + WWPGD

Matching seeds use the same model, data, minibatch stream, schedule, and evaluation probes.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
BASELINE = Path(os.getenv('NANOGPT_LEVEL0_BASELINE_RESULTS_ROOT', '/tmp/nanogpt-level0-bpe/results'))
WWPGD = Path(os.getenv('NANOGPT_LEVEL0_WWPGD_RESULTS_ROOT', '/tmp/nanogpt-level0-wwpgd/results'))
rows=[]; selected=[]
for label, root, pattern in [('adamw', BASELINE, 'adamw_seed_*'), ('adamw_wwpgd', WWPGD, 'adamw_wwpgd_seed_*')]:
    for run in sorted(root.glob(pattern)):
        seed=int(run.name.rsplit('_seed_',1)[1]); d=pd.read_csv(run/'metrics.csv'); d['arm']=label; d['seed']=seed; rows.append(d)
        p=run/'selected_checkpoint_metrics.json'
        if p.exists(): x=json.loads(p.read_text()); x.update(arm=label, seed=seed); selected.append(x)
all_df=pd.concat(rows, ignore_index=True); selected_df=pd.DataFrame(selected)
sorted(set(all_df.seed)), selected_df


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
for arm, d in all_df.groupby('arm'):
    a=d.groupby('step').val_loss.agg(['mean','std']).reset_index(); s=a['std'].fillna(0); line,=ax.plot(a.step,a['mean'],label=arm); ax.fill_between(a.step,a['mean']-s,a['mean']+s,alpha=.2,color=line.get_color())
ax.set(xlabel='optimizer step', ylabel='validation loss', title='AdamW versus AdamW + WWPGD: mean +/- 1 std')
ax.grid(alpha=.25); ax.legend(); plt.show()


In [ ]:
pivot=selected_df.pivot(index='seed', columns='arm', values='test_loss').dropna()
pivot['wwpgd_minus_adamw']=pivot['adamw_wwpgd']-pivot['adamw']
display(pivot)
print('mean paired test-loss difference:', pivot.wwpgd_minus_adamw.mean())
print('seeds improved:', int((pivot.wwpgd_minus_adamw < 0).sum()), '/', len(pivot))
